# 07 — CAD Export

Export the best design for manufacturing:
1. STL mesh (body + wing)
2. Airfoil profiles (.dat Selig format)
3. LE/TE splines for CAD loft
4. Assembly JSON for PicoGK pipeline
5. EDF duct positioning
6. Design report

In [29]:
import sys
sys.path.insert(0, '..')

import numpy as np
from pathlib import Path

from src.parameterization.design_variables import BWBParams, params_from_vector
from src.parameterization.bwb_aircraft import (
    build_airplane, compute_wing_area, compute_aspect_ratio, compute_mac,
    compute_internal_volume, estimate_structural_mass,
)
from src.aero.evaluator import AeroEvaluator
from src.aero.mission import MissionCondition
from src.propulsion.edf_model import EDF_70MM, duct_fits_in_body

In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load Best Design

In [31]:
try:
    best_x = np.load('../output/best_x.npy')
    params = params_from_vector(best_x)
    print('Loaded optimized design')
except FileNotFoundError:
    params = BWBParams()
    print('Using default params')

mission = MissionCondition()
result = AeroEvaluator(mission).evaluate(params)

Loaded optimized design


## 2. Export STL

In [32]:
from src.visualization.export import export_aircraft_stl, export_aircraft_step

out_dir = Path('../output/cad_export')
out_dir.mkdir(parents=True, exist_ok=True)

# STL (triangulated mesh, for visualization)
n_tri = export_aircraft_stl(params, str(out_dir / 'neuron_v2.stl'))
print(f'STL exported: {n_tri} triangles → {out_dir}/neuron_v2.stl')

# STEP (NURBS surfaces, for CAD/manufacturing)
step_path = export_aircraft_step(params, str(out_dir / 'neuron_v2.step'))
import os
step_kb = os.path.getsize(step_path) / 1024
print(f'STEP exported: {step_kb:.0f} KB → {step_path}')

STL exported: 15045 triangles → ..\output\cad_export/neuron_v2.stl
STEP exported: 1526 KB → ..\output\cad_export\neuron_v2.step


## 2b. STEP v2 — Watertight Solid (ThruSections loft)

In [ ]:
from src.visualization.export import export_aircraft_step_v2

metrics = export_aircraft_step_v2(params, str(out_dir / 'neuron_v2_solid.step'))

print(f'STEP v2 exported: {metrics["file_size_kb"]:.0f} KB → {metrics["path"]}')
print(f'  Valid solid  : {metrics["is_valid"]}')
print(f'  Volume       : {metrics["volume_mm3"]/1e3:.1f} cm³')
print(f'  Surface area : {metrics["surface_area_mm2"]/1e2:.0f} cm²')
print(f'  Max deviation: {metrics["max_deviation_mm"]:.4f} mm')
print(f'  Sections used: {metrics["n_sections"]}')

## 3. Export CAD Profiles

In [33]:
from src.visualization.export import export_cad_profiles

assembly = export_cad_profiles(params, str(out_dir))
print(f'CAD profiles exported to {out_dir}/')
print(f'Sections: {len(assembly["sections"])}')

CAD profiles exported to ..\output\cad_export/
Sections: 9


## 4. EDF Positioning

In [34]:
print('═══ EDF Integration ═══')
body_chord = params.body_root_chord
body_height = params.body_tc_root * body_chord
print(f'  Body chord       : {body_chord*1000:.0f} mm')
print(f'  Body height (max): {body_height*1000:.1f} mm')
print(f'  EDF duct OD      : {EDF_70MM.duct_outer_diameter*1000:.0f} mm')
print(f'  Duct fits        : {duct_fits_in_body(params.body_tc_root, body_chord, EDF_70MM)}')
print(f'  EDF position X   : {body_chord*0.45*1000:.0f} mm from nose (45% chord)')
print(f'  Thrust axis Z    : 0 mm (aligned with CG)')

═══ EDF Integration ═══
  Body chord       : 700 mm
  Body height (max): 151.4 mm
  EDF duct OD      : 78 mm
  Duct fits        : True
  EDF position X   : 315 mm from nose (45% chord)
  Thrust axis Z    : 0 mm (aligned with CG)


## 5. Design Report

In [35]:
print('\n═══════════════════════════════════════════')
print('       nEUROn v2 — Final Design Report')
print('═══════════════════════════════════════════')
print(f'\nGeometry:')
print(f'  Span             : {2*params.half_span:.2f} m')
print(f'  Wing root chord  : {params.wing_root_chord*1000:.0f} mm')
print(f'  Body chord       : {params.body_root_chord*1000:.0f} mm')
print(f'  Body width       : {2*params.body_halfwidth*1000:.0f} mm')
print(f'  Taper ratio      : {params.taper_ratio}')
print(f'  LE sweep (wing)  : {params.le_sweep_deg}°')
print(f'  LE sweep (body)  : {params.body_sweep_deg}°')
print(f'  Wing area        : {compute_wing_area(params)*1e4:.0f} cm²')
print(f'  Aspect ratio     : {compute_aspect_ratio(params):.2f}')
print(f'\nAerodynamics:')
print(f'  L/D              : {result["L_over_D"]:.2f}')
print(f'  CL               : {result["CL"]:.4f}')
print(f'  CD               : {result["CD"]:.5f}')
print(f'  CD0 wing         : {result["CD0_wing"]:.5f}')
print(f'  CD0 body         : {result["CD0_body"]:.5f}')
print(f'  CDi              : {result["CDi"]:.5f}')
print(f'  Static margin    : {result["static_margin"]:.1%} MAC')
print(f'  Cn_beta          : {result["Cn_beta"]:.4f}')
print(f'\nStructure:')
print(f'  Struct mass      : {result["struct_mass"]*1000:.0f} g')
print(f'  Internal volume  : {result["internal_volume"]*1e6:.0f} cm³')
print(f'\nPropulsion:')
print(f'  T/D              : {result["T_over_D"]:.2f}')
print(f'  P_elec           : {result["P_elec"]:.1f} W')
print(f'  Endurance        : {result["endurance_min"]:.1f} min')
print(f'  Range            : {result["range_km"]:.1f} km')


═══════════════════════════════════════════
       nEUROn v2 — Final Design Report
═══════════════════════════════════════════

Geometry:
  Span             : 2.00 m
  Wing root chord  : 352 mm
  Body chord       : 700 mm
  Body width       : 160 mm
  Taper ratio      : 0.2782344590817651
  LE sweep (wing)  : 20.61577308703825°
  LE sweep (body)  : 29.07885342319942°
  Wing area        : 4976 cm²
  Aspect ratio     : 8.02

Aerodynamics:
  L/D              : 22.87
  CL               : 0.4142
  CD               : 0.01811
  CD0 wing         : 0.00424
  CD0 body         : 0.00541
  CDi              : 0.00846
  Static margin    : 9.0% MAC
  Cn_beta          : 0.4497

Structure:
  Struct mass      : 825 g
  Internal volume  : 2051 cm³

Propulsion:
  T/D              : 1.57
  P_elec           : 116.9 W
  Endurance        : 8.0 min
  Range            : 9.6 km
